<a href="https://colab.research.google.com/github/ZuhaaAsif/Applied-Search-Intelligence-Google-Search-Ranking-Discoverability/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZuhaaAsif/Applied-Search-Intelligence-Google-Search-Ranking-Discoverability/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
# Setup
%pip -q install duckdb huggingface_hub

import os, getpass
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH_PATH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

data = con.sql(f"""
    WITH bounds AS (SELECT MIN(report_date) AS start_d FROM read_parquet('{MONTH_PATH}')),
    windowed AS (
        SELECT client_hash_id, content_hash_id,
               SUM(CASE WHEN report_date <= b.start_d + INTERVAL 15 DAY THEN gsc_impressions ELSE 0 END) AS imp_first_half,
               SUM(CASE WHEN report_date >  b.start_d + INTERVAL 15 DAY THEN gsc_impressions ELSE 0 END) AS imp_second_half,
               SUM(CASE WHEN report_date <= b.start_d + INTERVAL 15 DAY THEN gsc_clicks ELSE 0 END) AS clk_first_half,
               SUM(CASE WHEN report_date >  b.start_d + INTERVAL 15 DAY THEN gsc_clicks ELSE 0 END) AS clk_second_half,
               SUM(CASE WHEN report_date <= b.start_d + INTERVAL 15 DAY AND gsc_sum_position >= gsc_impressions THEN gsc_sum_position ELSE 0 END) AS sum_pos_fh,
               SUM(CASE WHEN report_date <= b.start_d + INTERVAL 15 DAY AND gsc_sum_position >= gsc_impressions THEN gsc_impressions ELSE 0 END) AS imp_with_pos_fh
        FROM read_parquet('{MONTH_PATH}') f, bounds b
        GROUP BY 1, 2 HAVING imp_first_half >= 100
    ) SELECT * FROM windowed
""").df()

data['pos_first_half']  = data['sum_pos_fh'] / data['imp_with_pos_fh']
data['ctr_first_half']  = data['clk_first_half'] / data['imp_first_half']
data['ctr_second_half'] = data['clk_second_half'] / data['imp_second_half'].replace(0, np.nan)

content_meta = con.sql(f"SELECT content_hash_id, content_type, content_created_date FROM read_parquet('{REL}/dim_content.parquet')").df()
data = data.merge(content_meta, on='content_hash_id', how='left')
data['content_age_days'] = (pd.Timestamp('2026-03-31') - pd.to_datetime(data['content_created_date'])).dt.days
data = data[data['pos_first_half'].notna()].copy()
data['position_bin'] = pd.cut(data['pos_first_half'], bins=[0,3,10,20,50,100000], labels=['1-3','4-10','11-20','21-50','51+'])

modelable = data[(data['imp_second_half'] >= 30) & data['ctr_second_half'].notna()].copy()
tier_2h = modelable.groupby('position_bin', observed=True).apply(lambda g: g['clk_second_half'].sum()/g['imp_second_half'].sum(), include_groups=False)
modelable['tier_expected_ctr_2h'] = modelable['position_bin'].map(tier_2h).astype(float)
modelable['is_underperformer'] = (modelable['ctr_second_half'] < modelable['tier_expected_ctr_2h']).astype(int)
print(f"{len(modelable):,} modelable rows, base rate {modelable['is_underperformer'].mean():.3f}")

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

78,431 modelable rows, base rate 0.701


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

CTR by Position. The paper reports CTR by position tier (Top 3 = 0.42%, down to Deep = 0.05%). This is close to my own work, so I have a specific question: does the paper's position number get cleaned the same way mine had to be? In my own March data, I found that about 2.7% of rows (affecting 38.7% of pages) had impossible position values, a page could show an average position under 1, which can't really happen. If I hadn't caught and fixed that, some pages would have been placed in the wrong tier. I'm not saying the paper has this problem, I don't know how they built it, but since it's the same warehouse, it's the first thing I'd want checked before trusting this finding fully.

Zombie Recovery model. This model predicts whether a dead page comes back to life, and it scores 99% accuracy on known brands and 97% on brand-new ones. That's a very high score, and the leakage skill says high scores like this should be checked, not just trusted. Two questions: First, is there a clean gap in time between the features used and the outcome being predicted, so the model isn't accidentally peeking at the answer? Second, how were the pages picked for this test, if only pages with some prior history were included, that's a reasonable choice, but it should be stated clearly, since it could make the model look better than it would on totally new pages with no history at all.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5 model already used a grouped split (splitting by client, not just randomly), which is the honest way to test it. To actually show what that grouping does, I built a plain random split on purpose, letting the same client show up in both training and testing, and compared it side by side with the honest grouped split.

Results: Random split scored 0.90 precision@20 and 0.92 precision@50. Grouped split scored 0.95 precision@20 and 0.90 precision@50. Normally we'd expect the random split to score higher, since it can cheat a little by seeing the same client twice. That's not really what happened here, the grouped split actually did slightly better at @20. The most likely reason is that the grouped test only had 10 clients in it, which is a small number, so the score can jump around a bit just by chance. This is a real limit of using one month of data with only around 40 clients total, not a sign that something is broken.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

num_cols = ['imp_first_half', 'pos_first_half', 'ctr_first_half', 'content_age_days']
cat_cols = ['content_type']
preprocess = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)], remainder='passthrough')

def run_split(train_idx, test_idx, label):
    train, test = modelable.iloc[train_idx], modelable.iloc[test_idx]
    X_tr, y_tr = train[num_cols + cat_cols], train['is_underperformer']
    X_te, y_te = test[num_cols + cat_cols], test['is_underperformer']
    rf = Pipeline([('prep', preprocess), ('clf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))]).fit(X_tr, y_tr)
    scores = rf.predict_proba(X_te)[:, 1]
    return {
        'split': label,
        'precision@20': precision_at_k(scores, y_te.values, 20),
        'precision@50': precision_at_k(scores, y_te.values, 50),
        'base_rate': y_te.mean(),
        'test_clients': test['client_hash_id'].nunique(),
    }

# BEFORE: naive random split — same client can appear in both sides
rand_train_idx, rand_test_idx = train_test_split(np.arange(len(modelable)), test_size=0.25, random_state=42, stratify=modelable['is_underperformer'])
before = run_split(rand_train_idx, rand_test_idx, 'Random (naive)')

# AFTER: honest client-grouped split — same as Week 5
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
grp_train_idx, grp_test_idx = next(gss.split(modelable, groups=modelable['client_hash_id']))
after = run_split(grp_train_idx, grp_test_idx, 'Grouped by client (honest)')

pd.DataFrame([before, after])

,split,precision@20,precision@50,base_rate,test_clients
0,Random (naive),0.90,0.92,0.700734,33
1,Grouped by client (honest),0.95,0.90,0.685388,10


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I checked my final model against the leakage checklist:

1. All features come from before the outcome is known, and the label comes from strictly after; no overlap.
2. I tested this directly by deliberately adding the "answer" column back in.
3. No FlyRank product scores or flags were used anywhere.
4. The filters I used to pick which pages to include are based only on how much traffic a page had, not on whether it went up or down, so they can't be secretly leaking the answer.
5. The split is grouped by client, as shown in Section 2.
6. The base rate (0.701 overall) is printed and compared against every score.

The leak test: When I put the honest features in, the model scored 0.950 precision@20. When I deliberately added back the column the label is built from (ctr_second_half), the score jumped to 1.000; a perfect score. This is a good sign, not a bad one: it proves my testing setup actually catches leakage when it's there, which means I can trust that my normal, honest model doesn't have this problem.

What the model actually leans on: the biggest factor is ctr_first_half (0.348), then position (0.262), then impressions (0.242), then content age (0.146). The type of content barely mattered at all (under 0.001). All of these are things that would genuinely be known before the outcome, so nothing here looks suspicious.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
leak_train, leak_test = modelable.iloc[grp_train_idx].copy(), modelable.iloc[grp_test_idx].copy()
leak_cols = num_cols + cat_cols + ['ctr_second_half']

X_tr_h, y_tr = leak_train[num_cols + cat_cols], leak_train['is_underperformer']
X_te_h, y_te = leak_test[num_cols + cat_cols], leak_test['is_underperformer']
honest_rf = Pipeline([('prep', preprocess), ('clf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))]).fit(X_tr_h, y_tr)
honest_p20 = precision_at_k(honest_rf.predict_proba(X_te_h)[:,1], y_te.values, 20)

X_tr_l, X_te_l = leak_train[leak_cols], leak_test[leak_cols]
leak_prep = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)], remainder='passthrough')
leaky_rf = Pipeline([('prep', leak_prep), ('clf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))]).fit(X_tr_l, y_tr)
leaky_p20 = precision_at_k(leaky_rf.predict_proba(X_te_l)[:,1], y_te.values, 20)

print(f"Honest precision@20: {honest_p20:.3f}")
print(f"Leaky (ctr_second_half included) precision@20: {leaky_p20:.3f}")

importances = honest_rf.named_steps['clf'].feature_importances_
names = honest_rf.named_steps['prep'].get_feature_names_out()
print(pd.DataFrame({'feature': names, 'importance': importances}).sort_values('importance', ascending=False))

Honest precision@20: 0.950
Leaky (ctr_second_half included) precision@20: 1.000
                                feature  importance
5             remainder__ctr_first_half    0.348263
4             remainder__pos_first_half    0.261574
3             remainder__imp_first_half    0.241562
6           remainder__content_age_days    0.146422
0  cat__content_type_comparison article    0.000787
2     cat__content_type_keyword article    0.000761
1      cat__content_type_feedly article    0.000631


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

What I said before: "That's a genuine, persistent pattern in CTR behavior across the month, not an artifact."

This sounds more certain than it should. It was based on one month of data and one specific way of splitting it, not something proven to hold everywhere.

Safer version: "In this March 2026 sample, pages that were already underperforming in the first half of the month were more likely to still be underperforming in the second half (81% of the time) compared to pages that weren't (45% of the time). This points toward a real pattern worth acting on, but it hasn't been tested on a different month or in a more rigorous experiment, so it should be treated as a helpful signal, not a proven fact."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.